In [3]:
from pathlib import Path
import re
import json
import shutil
import subprocess

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

from nibabel.processing import resample_from_to
from ipywidgets import interact, IntSlider, Dropdown, FloatSlider, Checkbox

In [4]:
base = Path("/media/bami/JHA/rsCVR/data/Tests")

SUBJECT = "Test_1a"
# SUBJECT = "Test_2"
# SUBJECT = "Test_3"
# SUBJECT = "Test_4"
# SUBJECT = "Test_5"

TR = 0.5  # seconds

# Output control
FORCE_REBUILD_BOLD4D = False
FORCE_RERUN_RAPIDTIDE = False

# Rapidtide settings
SEARCH_RANGE_NARROW = (-10, 30)
SEARCH_RANGE_WIDE = (-15, 45)

# Stable lag mask criteria
LAG_DIFF_THR = 5.0
STABLE_LAG_MIN = -10.0
STABLE_LAG_MAX = 35.0
MAXCORR_THR = 0.20
MTT_MIN = 0.1
MTT_MAX = 6.0

# Lag-corrected GLM settings
LAG_SIGN = -1
R2_THR = 0.05
MIN_VALID_FRAC = 0.70

print("Subject:", SUBJECT)
print("Base:", base)
print("TR:", TR)

Subject: Test_1a
Base: /media/bami/JHA/rsCVR/data/Tests
TR: 0.5


In [5]:
# Defs

def natural_key(path):
    name = Path(path).name
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", name)
    ]


def get_volume_index(path):
    name = Path(path).name
    m = re.search(r"_(\d+)\.nii(\.gz)?$", name)
    if m:
        return int(m.group(1))
    return None


def find_one(directory, pattern, description):
    files = sorted(directory.glob(pattern), key=natural_key)

    if len(files) == 0:
        raise FileNotFoundError(f"{description} file not found: {directory / pattern}")

    if len(files) > 1:
        print(f"[Warning] Multiple {description} files found. Using the first one.")
        for f in files:
            print("  -", f)

    return files[0]


def load_nifti(path):
    return nib.load(str(path))


def same_grid(img1, img2, affine_tol=1e-4):
    same_shape = img1.shape[:3] == img2.shape[:3]
    same_affine = np.allclose(img1.affine, img2.affine, atol=affine_tol)
    return same_shape and same_affine


def make_3d_header(ref_img, dtype=np.float32):
    header = ref_img.header.copy()
    header.set_data_shape(ref_img.shape[:3])
    header.set_zooms(ref_img.header.get_zooms()[:3])
    header.set_data_dtype(dtype)
    return header


def save_mask(mask_bool, ref_img, out_path):
    header = make_3d_header(ref_img, dtype=np.uint8)
    img = nib.Nifti1Image(
        mask_bool.astype(np.uint8),
        ref_img.affine,
        header
    )
    img.set_qform(ref_img.affine, code=2)
    img.set_sform(ref_img.affine, code=2)
    nib.save(img, str(out_path))


def save_float_map(data, ref_img, out_path):
    header = make_3d_header(ref_img, dtype=np.float32)
    img = nib.Nifti1Image(
        data.astype(np.float32),
        ref_img.affine,
        header
    )
    img.set_qform(ref_img.affine, code=2)
    img.set_sform(ref_img.affine, code=2)
    nib.save(img, str(out_path))


def make_4d_from_3d_files(files, out_path, tr):
    first_img = load_nifti(files[0])
    first_shape = first_img.shape[:3]
    first_affine = first_img.affine

    n_vol = len(files)
    data_4d = np.empty(first_shape + (n_vol,), dtype=np.float32)

    for i, f in enumerate(files):
        img = load_nifti(f)

        if img.shape[:3] != first_shape:
            raise ValueError(
                f"Shape mismatch at {f}\n"
                f"Expected: {first_shape}, got: {img.shape[:3]}"
            )

        if not np.allclose(img.affine, first_affine, atol=1e-4):
            raise ValueError(
                f"Affine mismatch at {f}\n"
                "All swra images must have the same affine."
            )

        data_4d[..., i] = np.asarray(img.dataobj, dtype=np.float32)

    out_img = nib.Nifti1Image(data_4d, first_affine)
    out_img.header.set_data_dtype(np.float32)
    spatial_zooms = first_img.header.get_zooms()[:3]
    out_img.header.set_zooms(spatial_zooms + (tr,))

    qform_code = int(np.asarray(first_img.header["qform_code"]))
    sform_code = int(np.asarray(first_img.header["sform_code"]))

    if qform_code == 0:
        qform_code = 2
    if sform_code == 0:
        sform_code = 2

    out_img.set_qform(first_affine, code=qform_code)
    out_img.set_sform(first_affine, code=sform_code)

    nib.save(out_img, str(out_path))
    return out_img


def resample_prob_to_ref(prob_img, ref_img, order=1):
    return resample_from_to(
        prob_img,
        (ref_img.shape[:3], ref_img.affine),
        order=order
    )


def run_command(cmd, log_path):
    print("\n[Command]")
    print(" ".join(map(str, cmd)))
    print()

    with open(log_path, "w") as f:
        f.write(" ".join(map(str, cmd)) + "\n\n")
        proc = subprocess.run(
            cmd,
            stdout=f,
            stderr=subprocess.STDOUT,
            text=True
        )

    if proc.returncode != 0:
        raise RuntimeError(
            f"Command failed with return code {proc.returncode}\n"
            f"Log file: {log_path}"
        )

    return proc.returncode


def check_rapidtide_log(log_path):
    bad_keywords = [
        "ERROR:",
        "no voxels qualify",
        "Refine mask has 0 voxels",
        "No voxels in refine mask",
        "exiting"
    ]

    with open(log_path, "r") as f:
        txt = f.read()

    found = [k for k in bad_keywords if k in txt]

    if found:
        print("[Warning] Potential rapidtide failure detected:")
        for k in found:
            print(" -", k)
    else:
        print("[OK] No obvious rapidtide failure keyword found.")

In [6]:
# Define paths and find input files

subj_dir = base / SUBJECT
rsfmri_dir = subj_dir / "RSfMRI"

smooth_dir = rsfmri_dir / "5_Smooth"
seg_dir = rsfmri_dir / "Segmentation"

out_dir = subj_dir / "RapidTide"
out_dir.mkdir(parents=True, exist_ok=True)

print("Subject dir:", subj_dir)
print("RSfMRI dir :", rsfmri_dir)
print("Smooth dir :", smooth_dir)
print("Seg dir    :", seg_dir)
print("Output dir :", out_dir)

# Required Dirs
for d in [subj_dir, rsfmri_dir, smooth_dir, seg_dir]:
    if not d.exists():
        raise FileNotFoundError(f"Missing required directory: {d}")

# Rapidtide executability
RAPIDTIDE_EXE = shutil.which("rapidtide")
if RAPIDTIDE_EXE is None:
    RAPIDTIDE_EXE = "/home/bami/anaconda3/envs/JHA-spmCVR/bin/rapidtide"

if not Path(RAPIDTIDE_EXE).exists():
    raise FileNotFoundError(f"rapidtide executable not found: {RAPIDTIDE_EXE}")

print("rapidtide:", RAPIDTIDE_EXE)

# Find 3D BOLD and segmentation files
smooth_files = sorted(
    list(smooth_dir.glob("swra*_RSfMRI_TR*_*.nii")) +
    list(smooth_dir.glob("swra*_RSfMRI_TR*_*.nii.gz")),
    key=natural_key
)

if len(smooth_files) == 0:
    raise FileNotFoundError(f"No swra RSfMRI files found in: {smooth_dir}")

vol_indices = [get_volume_index(f) for f in smooth_files]
if all(v is not None for v in vol_indices):
    expected = list(range(min(vol_indices), min(vol_indices) + len(vol_indices)))
    if vol_indices != expected:
        print("[Warning] Volume indices may not be continuous.")
        print("First 10:", vol_indices[:10])
        print("Last 10 :", vol_indices[-10:])

gm_path = find_one(seg_dir, "c1wr*_*_3DT1.nii*", "GM c1")
wm_path = find_one(seg_dir, "c2wr*_*_3DT1.nii*", "WM c2")
csf_path = find_one(seg_dir, "c3wr*_*_3DT1.nii*", "CSF c3")

print("Number of BOLD volumes:", len(smooth_files))
print("First:", smooth_files[0].name)
print("Last :", smooth_files[-1].name)
print("GM  :", gm_path.name)
print("WM  :", wm_path.name)
print("CSF :", csf_path.name)

Subject dir: /media/bami/JHA/rsCVR/data/Tests/Test_1a
RSfMRI dir : /media/bami/JHA/rsCVR/data/Tests/Test_1a/RSfMRI
Smooth dir : /media/bami/JHA/rsCVR/data/Tests/Test_1a/RSfMRI/5_Smooth
Seg dir    : /media/bami/JHA/rsCVR/data/Tests/Test_1a/RSfMRI/Segmentation
Output dir : /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide
rapidtide: /home/bami/anaconda3/envs/JHA-spmCVR/bin/rapidtide
Number of BOLD volumes: 600
First: swraYMS_RSfMRI_TR500_00001.nii
Last : swraYMS_RSfMRI_TR500_00600.nii
GM  : c1wrflipLR_YMS_3DT1.nii
WM  : c2wrflipLR_YMS_3DT1.nii
CSF : c3wrflipLR_YMS_3DT1.nii


In [7]:
# Make 4D BOLD
bold4d_path = out_dir / f"{SUBJECT}_desc-SPMpreprocSmooth_bold4d.nii.gz"

if bold4d_path.exists() and not FORCE_REBUILD_BOLD4D:
    print("[Load existing 4D BOLD]")
    bold4d_img = nib.load(str(bold4d_path))
else:
    print("[Making 4D BOLD]")
    bold4d_img = make_4d_from_3d_files(
        files=smooth_files,
        out_path=bold4d_path,
        tr=TR
    )

print("Saved:", bold4d_path)
print("4D BOLD shape:", bold4d_img.shape)
print("4D BOLD zooms:", bold4d_img.header.get_zooms())


# Reload check
check_img = nib.load(str(bold4d_path))
print("Reloaded shape:", check_img.shape)
print("Reloaded zooms:", check_img.header.get_zooms())
print("qform code:", int(np.asarray(check_img.header["qform_code"])))
print("sform code:", int(np.asarray(check_img.header["sform_code"])))

[Making 4D BOLD]
Saved: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-SPMpreprocSmooth_bold4d.nii.gz
4D BOLD shape: (79, 95, 79, 600)
4D BOLD zooms: (np.float32(2.0), np.float32(2.0), np.float32(2.0), np.float32(0.5))
Reloaded shape: (79, 95, 79, 600)
Reloaded zooms: (np.float32(2.0), np.float32(2.0), np.float32(2.0), np.float32(0.5))
qform code: 2
sform code: 2


In [8]:
# Load segmentation and create masks

print("[Making masks]")

gm_img = nib.load(str(gm_path))
wm_img = nib.load(str(wm_path))
csf_img = nib.load(str(csf_path))

if not same_grid(gm_img, wm_img):
    raise ValueError("GM and WM grids are different.")
if not same_grid(gm_img, csf_img):
    raise ValueError("GM and CSF grids are different.")

if same_grid(gm_img, bold4d_img):
    print("Segmentation grid == BOLD grid. Resampling skipped.")
    gm_ref_img = gm_img
    wm_ref_img = wm_img
    csf_ref_img = csf_img
else:
    print("Segmentation grid != BOLD grid.")
    print("Resampling segmentation probability maps to BOLD grid.")
    gm_ref_img = resample_prob_to_ref(gm_img, bold4d_img, order=1)
    wm_ref_img = resample_prob_to_ref(wm_img, bold4d_img, order=1)
    csf_ref_img = resample_prob_to_ref(csf_img, bold4d_img, order=1)

gm = gm_ref_img.get_fdata(dtype=np.float32)
wm = wm_ref_img.get_fdata(dtype=np.float32)
csf = csf_ref_img.get_fdata(dtype=np.float32)

brain_mask = (gm + wm + csf) > 0.5
matter_mask = (gm + wm) > 0.5

gm_mask = gm > 0.5
wm_mask = wm > 0.5
csf_mask = csf > 0.5

brain_mask_path = out_dir / f"{SUBJECT}_desc-brain_mask.nii.gz"
matter_mask_path = out_dir / f"{SUBJECT}_desc-matter_mask.nii.gz"

gm_mask_path = out_dir / f"{SUBJECT}_desc-GM_mask.nii.gz"
wm_mask_path = out_dir / f"{SUBJECT}_desc-WM_mask.nii.gz"
csf_mask_path = out_dir / f"{SUBJECT}_desc-CSF_mask.nii.gz"

save_mask(brain_mask, bold4d_img, brain_mask_path)
save_mask(matter_mask, bold4d_img, matter_mask_path)
save_mask(gm_mask, bold4d_img, gm_mask_path)
save_mask(wm_mask, bold4d_img, wm_mask_path)
save_mask(csf_mask, bold4d_img, csf_mask_path)

print("Saved:", brain_mask_path)
print("Saved:", matter_mask_path)
print("Saved:", gm_mask_path)
print("Saved:", wm_mask_path)
print("Saved:", csf_mask_path)

print("Brain voxels :", int(brain_mask.sum()))
print("Matter voxels:", int(matter_mask.sum()))
print("GM voxels    :", int(gm_mask.sum()))
print("WM voxels    :", int(wm_mask.sum()))
print("CSF voxels   :", int(csf_mask.sum()))

[Making masks]
Segmentation grid == BOLD grid. Resampling skipped.
Saved: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-brain_mask.nii.gz
Saved: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-matter_mask.nii.gz
Saved: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-GM_mask.nii.gz
Saved: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-WM_mask.nii.gz
Saved: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-CSF_mask.nii.gz
Brain voxels : 234852
Matter voxels: 159942
GM voxels    : 85823
WM voxels    : 73077
CSF voxels   : 72577


In [9]:
# Optional viewer: BOLD and masks
bold_img = nib.load(str(bold4d_path))
bold_data = bold_img.get_fdata(dtype=np.float32)

nx, ny, nz, nt = bold_data.shape

brain_mask_data = nib.load(str(brain_mask_path)).get_fdata() > 0
matter_mask_data = nib.load(str(matter_mask_path)).get_fdata() > 0
gm_mask_data = nib.load(str(gm_mask_path)).get_fdata() > 0

def show_mask_overlay(
    mask_type="matter",
    volume=1,
    z=nz//2,
    alpha=0.4,
    bg_vmin_pct=1,
    bg_vmax_pct=99
):
    bg = bold_data[..., volume - 1]

    if mask_type == "brain":
        mask = brain_mask_data
    elif mask_type == "matter":
        mask = matter_mask_data
    elif mask_type == "GM":
        mask = gm_mask_data
    else:
        raise ValueError("mask_type must be brain, matter, or GM")

    bg2d = bg[:, :, z].T
    mask2d = mask[:, :, z].T

    bg_vmin, bg_vmax = np.percentile(bg[np.isfinite(bg)], [bg_vmin_pct, bg_vmax_pct])

    plt.figure(figsize=(6, 6))
    plt.imshow(bg2d, cmap="gray", origin="lower", vmin=bg_vmin, vmax=bg_vmax)

    overlay = np.ma.masked_where(mask2d == 0, mask2d)
    plt.imshow(overlay, cmap="autumn", origin="lower", alpha=alpha, vmin=0, vmax=1)

    plt.title(f"{SUBJECT} | {mask_type} mask | BG vol {volume}/{nt} | Axial z={z}")
    plt.axis("off")
    plt.show()

interact(
    show_mask_overlay,
    mask_type=Dropdown(options=["brain", "matter", "GM"], value="matter", description="Mask"),
    volume=IntSlider(value=1, min=1, max=nt, step=1, description="BG vol"),
    z=IntSlider(value=nz//2, min=0, max=nz-1, step=1, description="Z slice"),
    alpha=FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description="Alpha"),
    bg_vmin_pct=FloatSlider(value=1, min=0, max=10, step=0.5, description="BG min%"),
    bg_vmax_pct=FloatSlider(value=99, min=90, max=100, step=0.5, description="BG max%")
)

interactive(children=(Dropdown(description='Mask', index=1, options=('brain', 'matter', 'GM'), value='matter')…

<function __main__.show_mask_overlay(mask_type='matter', volume=1, z=39, alpha=0.4, bg_vmin_pct=1, bg_vmax_pct=99)>

In [10]:
# Run rapidtide: internal GM probe, single-pass lag mapping
def run_internal_gm_singlepass(search_range, label):
    rt_dir = out_dir / label

    if FORCE_RERUN_RAPIDTIDE and rt_dir.exists():
        print("[Delete existing rapidtide result dir]", rt_dir)
        shutil.rmtree(rt_dir)

    rt_dir.mkdir(parents=True, exist_ok=True)

    outroot = rt_dir / SUBJECT
    log_path = rt_dir / f"{SUBJECT}_rapidtide.log"

    cmd = [
        RAPIDTIDE_EXE,
        str(bold4d_path),
        str(outroot),

        "--datatstep", str(TR),

        "--brainmask", str(brain_mask_path),
        "--corrmask", str(matter_mask_path),

        "--globalmeaninclude", str(gm_mask_path),

        "--graymattermask", str(gm_mask_path),
        "--whitemattermask", str(wm_mask_path),
        "--csfmask", str(csf_mask_path),

        "--searchrange", str(search_range[0]), str(search_range[1]),
        "--passes", "1",

        "--nofitfilt",
        "--numnull", "0",
        "--outputlevel", "normal",
        "--spatialfilt", "-1",
    ]

    maxtime_path = rt_dir / f"{SUBJECT}_desc-maxtime_map.nii.gz"

    if maxtime_path.exists() and not FORCE_RERUN_RAPIDTIDE:
        print("[Skip existing rapidtide run]", rt_dir)
    else:
        run_command(cmd, log_path)
        check_rapidtide_log(log_path)

    return rt_dir, log_path


rt_result_dir_lag30, rt_log_path_lag30 = run_internal_gm_singlepass(
    SEARCH_RANGE_NARROW,
    "rapidtide_internalGMprobe_lag_singlepass_mattercorr_search-10to30"
)

rt_result_dir_lag45, rt_log_path_lag45 = run_internal_gm_singlepass(
    SEARCH_RANGE_WIDE,
    "rapidtide_internalGMprobe_lag_singlepass_mattercorr_search-15to45"
)

print("Lag30 dir:", rt_result_dir_lag30)
print("Lag45 dir:", rt_result_dir_lag45)


[Command]
/home/bami/anaconda3/envs/JHA-spmCVR/bin/rapidtide /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-SPMpreprocSmooth_bold4d.nii.gz /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rapidtide_internalGMprobe_lag_singlepass_mattercorr_search-10to30/Test_1a --datatstep 0.5 --brainmask /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-brain_mask.nii.gz --corrmask /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-matter_mask.nii.gz --globalmeaninclude /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-GM_mask.nii.gz --graymattermask /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-GM_mask.nii.gz --whitemattermask /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-WM_mask.nii.gz --csfmask /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-CSF_mask.nii.gz --searchrange -10 30 --passes 1 --nofitfilt --numnull 0 --outputlevel normal --spatialfilt -1

[OK] No obvious rapidtide failure key

In [11]:
# RapidTide QC functions

def map_stats(path, mask_path=None, name=None):
    img = nib.load(str(path))
    data = np.squeeze(img.get_fdata(dtype=np.float32))

    if mask_path is not None:
        mask = np.squeeze(nib.load(str(mask_path)).get_fdata()) > 0
        vals = data[mask]
    else:
        vals = data.ravel()

    vals = vals[np.isfinite(vals)]

    print("\n" + "=" * 80)
    print(name if name is not None else path.name)
    print("file:", path.name)
    print("n:", vals.size)

    if vals.size == 0:
        return

    print("min/max:", np.min(vals), np.max(vals))
    for p in [0.1, 1, 2, 5, 25, 50, 75, 95, 98, 99, 99.9]:
        print(f"p{p}: {np.percentile(vals, p)}")


def quick_rt_qc(rt_dir, subject):
    brainmask = nib.load(str(rt_dir / f"{subject}_desc-brainmask_mask.nii.gz")).get_fdata() > 0
    processed = nib.load(str(rt_dir / f"{subject}_desc-processed_mask.nii.gz")).get_fdata() > 0
    corrfit = nib.load(str(rt_dir / f"{subject}_desc-corrfit_mask.nii.gz")).get_fdata() > 0
    matter = nib.load(str(matter_mask_path)).get_fdata() > 0

    print("brainmask voxels:", int(brainmask.sum()))
    print("matter voxels   :", int(matter.sum()))
    print("processed voxels:", int(processed.sum()))
    print("corrfit voxels  :", int(corrfit.sum()))

    print("processed / brainmask:", processed.sum() / brainmask.sum())
    print("corrfit / brainmask  :", corrfit.sum() / brainmask.sum())
    print("processed / matter   :", processed.sum() / matter.sum())
    print("corrfit / matter     :", corrfit.sum() / matter.sum())
    print("corrfit / processed  :", corrfit.sum() / processed.sum())

    fail_path = rt_dir / f"{subject}_desc-corrfitfailreason_info.nii.gz"
    fail = np.squeeze(nib.load(str(fail_path)).get_fdata())
    vals = fail[brainmask]
    unique, counts = np.unique(vals.astype(int), return_counts=True)

    print("\nFailreason:")
    for u, c in zip(unique, counts):
        print(f"code {u}: {c} voxels ({c / brainmask.sum() * 100:.2f}%)")

In [12]:
# QC: narrow and wide lag mapping
for label, rt_dir in {
    "lag search -10 to 30": rt_result_dir_lag30,
    "lag search -15 to 45": rt_result_dir_lag45,
}.items():
    print("\n" + "#" * 100)
    print(label)
    print(rt_dir)
    print("#" * 100)

    quick_rt_qc(rt_dir, SUBJECT)

    for fname in [
        "maxtime_map",
        "maxtimerefined_map",
        "maxcorr_map",
        "maxcorrsq_map",
        "MTT_map",
    ]:
        map_stats(
            rt_dir / f"{SUBJECT}_desc-{fname}.nii.gz",
            rt_dir / f"{SUBJECT}_desc-corrfit_mask.nii.gz",
            name=f"{label} | {fname}"
        )


####################################################################################################
lag search -10 to 30
/media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rapidtide_internalGMprobe_lag_singlepass_mattercorr_search-10to30
####################################################################################################
brainmask voxels: 234852
matter voxels   : 159942
processed voxels: 153167
corrfit voxels  : 152368
processed / brainmask: 0.6521852060020779
corrfit / brainmask  : 0.6487830633760837
processed / matter   : 0.9576408948243739
corrfit / matter     : 0.9526453339335509
corrfit / processed  : 0.994783471635535

Failreason:
code 0: 234053 voxels (99.66%)
code 1024: 106 voxels (0.05%)
code 4096: 74 voxels (0.03%)
code 8192: 211 voxels (0.09%)
code 17412: 408 voxels (0.17%)

lag search -10 to 30 | maxtime_map
file: Test_1a_desc-maxtime_map.nii.gz
n: 152368
min/max: -10.163803 31.032652
p0.1: -9.40645694732666
p1: -7.913440227508545
p2: -6.897197723388672
p5

In [13]:
# Viewer: rapidtide lag maps
VIEW_RT_DIR = rt_result_dir_lag45

bold_img = nib.load(str(bold4d_path))
bold_data = bold_img.get_fdata(dtype=np.float32)
nx, ny, nz, nt = bold_data.shape


def collect_3d_maps(rt_dir, ref_shape):
    nii_files = sorted(
        list(rt_dir.rglob("*.nii")) + list(rt_dir.rglob("*.nii.gz")),
        key=lambda x: x.name
    )

    map_dict = {}

    for f in nii_files:
        img = nib.load(str(f))
        shape = img.shape
        squeezed_shape = tuple([s for s in shape if s != 1])

        if squeezed_shape == ref_shape:
            map_dict[f.name] = f

    return map_dict


rt_map_dict = collect_3d_maps(VIEW_RT_DIR, bold_data.shape[:3])
corrfit_view = np.squeeze(
    nib.load(str(VIEW_RT_DIR / f"{SUBJECT}_desc-corrfit_mask.nii.gz")).get_fdata()
) > 0

print("Viewing:", VIEW_RT_DIR)
print("3D maps:", len(rt_map_dict))
for name in rt_map_dict:
    print(name)


def load_rt_3d_map(map_name):
    path = rt_map_dict[map_name]
    data = np.squeeze(nib.load(str(path)).get_fdata(dtype=np.float32))

    if data.shape != bold_data.shape[:3]:
        raise ValueError(f"Shape mismatch: BOLD={bold_data.shape[:3]}, map={data.shape}")

    return data


def show_rt_overlay(
    map_name,
    volume=1,
    z=nz//2,
    alpha=0.65,
    bg_vmin_pct=1,
    bg_vmax_pct=99,
    map_pct_low=1,
    map_pct_high=99,
    mask_corrfit=True,
    symmetric=False
):
    bg = bold_data[..., volume - 1]
    data = load_rt_3d_map(map_name)

    if mask_corrfit and "mask" not in map_name.lower():
        data = data.copy()
        data[~corrfit_view] = np.nan

    bg2d = bg[:, :, z].T
    map2d = data[:, :, z].T

    bg_vmin, bg_vmax = np.percentile(bg[np.isfinite(bg)], [bg_vmin_pct, bg_vmax_pct])

    vals = data[np.isfinite(data)]
    vals = vals[vals != 0]

    if vals.size == 0:
        map_vmin, map_vmax = -1, 1
    elif "mask" in map_name.lower():
        map_vmin, map_vmax = 0, 1
    else:
        map_vmin, map_vmax = np.percentile(vals, [map_pct_low, map_pct_high])

    if symmetric and "mask" not in map_name.lower():
        vmax_abs = max(abs(map_vmin), abs(map_vmax))
        map_vmin, map_vmax = -vmax_abs, vmax_abs

    map2d_plot = np.ma.masked_where((~np.isfinite(map2d)) | (map2d == 0), map2d)

    plt.figure(figsize=(7, 7))
    plt.imshow(bg2d, cmap="gray", origin="lower", vmin=bg_vmin, vmax=bg_vmax)

    if "mask" in map_name.lower():
        cmap = "autumn"
    elif ("time" in map_name.lower()) or ("offset" in map_name.lower()):
        cmap = "coolwarm"
    else:
        cmap = "magma"

    im = plt.imshow(
        map2d_plot,
        cmap=cmap,
        origin="lower",
        alpha=alpha,
        vmin=map_vmin,
        vmax=map_vmax
    )

    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title(
        f"{SUBJECT} | rapidtide internal GM single-pass lag mapping\n"
        f"{map_name}\n"
        f"BG volume {volume}/{nt} | Axial z={z}"
    )
    plt.axis("off")
    plt.show()


default_map = f"{SUBJECT}_desc-maxtimerefined_map.nii.gz"

interact(
    show_rt_overlay,
    map_name=Dropdown(
        options=list(rt_map_dict.keys()),
        value=default_map if default_map in rt_map_dict else list(rt_map_dict.keys())[0],
        description="Map"
    ),
    volume=IntSlider(value=1, min=1, max=nt, step=1, description="BG vol"),
    z=IntSlider(value=nz//2, min=0, max=nz-1, step=1, description="Z slice"),
    alpha=FloatSlider(value=0.65, min=0.0, max=1.0, step=0.05, description="Alpha"),
    bg_vmin_pct=FloatSlider(value=1, min=0, max=10, step=0.5, description="BG min%"),
    bg_vmax_pct=FloatSlider(value=99, min=90, max=100, step=0.5, description="BG max%"),
    map_pct_low=FloatSlider(value=1, min=0, max=20, step=0.5, description="Map low%"),
    map_pct_high=FloatSlider(value=99, min=80, max=100, step=0.5, description="Map high%"),
    mask_corrfit=Checkbox(value=True, description="corrfit only"),
    symmetric=Checkbox(value=False, description="symmetric")
)

Viewing: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rapidtide_internalGMprobe_lag_singlepass_mattercorr_search-15to45
3D maps: 39
Test_1a_desc-CSF_mask.nii.gz
Test_1a_desc-CoV_map.nii.gz
Test_1a_desc-GM_mask.nii.gz
Test_1a_desc-MTT_map.nii.gz
Test_1a_desc-WM_mask.nii.gz
Test_1a_desc-brainmask_mask.nii.gz
Test_1a_desc-corrfit_mask.nii.gz
Test_1a_desc-corrfitfailreason_info.nii.gz
Test_1a_desc-datarange.nii.gz
Test_1a_desc-delayoffset_map.nii.gz
Test_1a_desc-filteredregressderivratios_map.nii.gz
Test_1a_desc-initialKurtosis_map.nii.gz
Test_1a_desc-initialSkewness_map.nii.gz
Test_1a_desc-initialStd_map.nii.gz
Test_1a_desc-initregressor_mask.nii.gz
Test_1a_desc-lfofilterCleanedKurtosis_map.nii.gz
Test_1a_desc-lfofilterCleanedSkewness_map.nii.gz
Test_1a_desc-lfofilterCleanedStd_map.nii.gz
Test_1a_desc-lfofilterCoeff_map.nii.gz
Test_1a_desc-lfofilterInbandVarianceAfter_map.nii.gz
Test_1a_desc-lfofilterInbandVarianceBefore_map.nii.gz
Test_1a_desc-lfofilterInbandVarianceChange_map.nii.

interactive(children=(Dropdown(description='Map', index=30, options=('Test_1a_desc-CSF_mask.nii.gz', 'Test_1a_…

<function __main__.show_rt_overlay(map_name, volume=1, z=39, alpha=0.65, bg_vmin_pct=1, bg_vmax_pct=99, map_pct_low=1, map_pct_high=99, mask_corrfit=True, symmetric=False)>

In [14]:
# Create stable lag-mask
lag30_path = rt_result_dir_lag30 / f"{SUBJECT}_desc-maxtimerefined_map.nii.gz"
lag45_path = rt_result_dir_lag45 / f"{SUBJECT}_desc-maxtimerefined_map.nii.gz"

corr30_path = rt_result_dir_lag30 / f"{SUBJECT}_desc-corrfit_mask.nii.gz"
corr45_path = rt_result_dir_lag45 / f"{SUBJECT}_desc-corrfit_mask.nii.gz"

lag30 = np.squeeze(nib.load(str(lag30_path)).get_fdata(dtype=np.float32))
lag45 = np.squeeze(nib.load(str(lag45_path)).get_fdata(dtype=np.float32))

corr30 = np.squeeze(nib.load(str(corr30_path)).get_fdata()) > 0
corr45 = np.squeeze(nib.load(str(corr45_path)).get_fdata()) > 0

matter = np.squeeze(nib.load(str(matter_mask_path)).get_fdata()) > 0

common = matter & corr30 & corr45 & np.isfinite(lag30) & np.isfinite(lag45)
diff = lag45 - lag30

vals = diff[common]
vals = vals[np.isfinite(vals)]

print("Common voxels:", int(common.sum()))
print("Common / matter:", common.sum() / matter.sum())

print("Lag difference: wide - narrow")
print("min/max:", vals.min(), vals.max())
print("p1, p5, p50, p95, p99:", np.percentile(vals, [1, 5, 50, 95, 99]))

for thr in [1, 2, 3, 5, 10]:
    stable = common & (np.abs(diff) <= thr)
    print(f"|diff| <= {thr:2d} s:", int(stable.sum()), f"({stable.sum() / matter.sum() * 100:.2f}% of matter)")


maxcorr45 = np.squeeze(
    nib.load(str(rt_result_dir_lag45 / f"{SUBJECT}_desc-maxcorr_map.nii.gz")).get_fdata(dtype=np.float32)
)

mtt45 = np.squeeze(
    nib.load(str(rt_result_dir_lag45 / f"{SUBJECT}_desc-MTT_map.nii.gz")).get_fdata(dtype=np.float32)
)

stable_lag_mask = (
    common
    & (np.abs(diff) <= LAG_DIFF_THR)
    & (lag45 >= STABLE_LAG_MIN)
    & (lag45 <= STABLE_LAG_MAX)
    & (maxcorr45 >= MAXCORR_THR)
    & (mtt45 >= MTT_MIN)
    & (mtt45 <= MTT_MAX)
)

stable_lag_mask_path = out_dir / f"{SUBJECT}_desc-stableLag_mask.nii.gz"
save_mask(stable_lag_mask, bold4d_img, stable_lag_mask_path)

print("\nStable lag voxels:", int(stable_lag_mask.sum()))
print("Stable / matter:", stable_lag_mask.sum() / matter.sum())
print("Saved:", stable_lag_mask_path)

Common voxels: 151977
Common / matter: 0.9502006977529355
Lag difference: wide - narrow
min/max: -44.986206 52.23703
p1, p5, p50, p95, p99: [-1.79282191e+01 -8.48025589e+00  5.53131104e-05  2.19816490e+01
  3.59048537e+01]
|diff| <=  1 s: 117998 (73.78% of matter)
|diff| <=  2 s: 119308 (74.59% of matter)
|diff| <=  3 s: 119827 (74.92% of matter)
|diff| <=  5 s: 120667 (75.44% of matter)
|diff| <= 10 s: 130782 (81.77% of matter)

Stable lag voxels: 98394
Stable / matter: 0.6151855047454702
Saved: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/Test_1a_desc-stableLag_mask.nii.gz


In [15]:
# Voxelwise lag-corrected GLM
glm_out_dir = out_dir / "rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag"
glm_out_dir.mkdir(parents=True, exist_ok=True)

lag_path = rt_result_dir_lag45 / f"{SUBJECT}_desc-maxtimerefined_map.nii.gz"

bold_img = nib.load(str(bold4d_path))
bold_data = bold_img.get_fdata(dtype=np.float32)

gm_mask_data = np.squeeze(nib.load(str(gm_mask_path)).get_fdata()) > 0
stable_lag_mask = np.squeeze(nib.load(str(stable_lag_mask_path)).get_fdata()) > 0
lag_map = np.squeeze(nib.load(str(lag_path)).get_fdata(dtype=np.float32))

nx, ny, nz, nt = bold_data.shape
t = np.arange(nt, dtype=np.float64) * TR

print("BOLD shape:", bold_data.shape)
print("Stable lag voxels:", int(stable_lag_mask.sum()))
print("LAG_SIGN:", LAG_SIGN)



# GM global mean regressor
gm_ts = bold_data[gm_mask_data, :]
gm_global_raw = np.nanmean(gm_ts, axis=0)

gm_global_percent = 100.0 * (gm_global_raw / np.nanmean(gm_global_raw) - 1.0)

reg_path = glm_out_dir / f"{SUBJECT}_desc-GMGlobalMean_percent_regressor.txt"
np.savetxt(reg_path, gm_global_percent, fmt="%.10f")

print("Saved regressor:", reg_path)
print("Regressor mean:", np.nanmean(gm_global_percent))
print("Regressor std :", np.nanstd(gm_global_percent))



# GLM helper functions
def shift_regressor_by_lag(reg, t, lag_sec, sign=-1):
    """
    sign = -1:
        x(t) = reg(t - lag)
        Positive lag means voxel response is delayed relative to the GM global regressor.

    sign = +1:
        x(t) = reg(t + lag)
    """
    shifted_time = t + sign * lag_sec

    return np.interp(
        shifted_time,
        t,
        reg,
        left=np.nan,
        right=np.nan
    )


def fit_simple_glm(y, x, min_valid=100):
    valid = np.isfinite(y) & np.isfinite(x)

    if valid.sum() < min_valid:
        return np.nan, np.nan, np.nan, np.nan, np.nan, valid.sum(), np.nan

    yy = y[valid].astype(np.float64)
    xx = x[valid].astype(np.float64)

    x_mean = xx.mean()
    y_mean = yy.mean()

    xc = xx - x_mean
    yc = yy - y_mean

    ssx = np.sum(xc ** 2)
    ssy = np.sum(yc ** 2)

    if ssx <= 0 or ssy <= 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan, valid.sum(), np.nan

    beta = np.sum(xc * yc) / ssx
    intercept = y_mean - beta * x_mean

    y_hat = intercept + beta * xx
    resid = yy - y_hat

    dof = valid.sum() - 2

    if dof > 0:
        resid_var = np.sum(resid ** 2) / dof
        resid_std = np.sqrt(resid_var)
        se_beta = np.sqrt(resid_var / ssx)
        t_beta = beta / se_beta if se_beta > 0 else np.nan
    else:
        resid_std = np.nan
        t_beta = np.nan

    r = np.sum(xc * yc) / np.sqrt(ssx * ssy)
    r2 = r ** 2

    return beta, intercept, r, r2, t_beta, valid.sum(), resid_std



# Run voxelwise GLM
MIN_VALID = int(nt * MIN_VALID_FRAC)

beta_map = np.full((nx, ny, nz), np.nan, dtype=np.float32)
intercept_map = np.full((nx, ny, nz), np.nan, dtype=np.float32)
r_map = np.full((nx, ny, nz), np.nan, dtype=np.float32)
r2_map = np.full((nx, ny, nz), np.nan, dtype=np.float32)
t_map = np.full((nx, ny, nz), np.nan, dtype=np.float32)
nvalid_map = np.zeros((nx, ny, nz), dtype=np.float32)
residstd_map = np.full((nx, ny, nz), np.nan, dtype=np.float32)

voxel_indices = np.array(np.where(stable_lag_mask)).T

print("Running GLM voxels:", len(voxel_indices))
print("MIN_VALID:", MIN_VALID)
print("LAG_SIGN:", LAG_SIGN)

for idx, (i, j, k) in enumerate(voxel_indices):
    if idx % 5000 == 0:
        print(f"{idx}/{len(voxel_indices)}")

    lag_sec = float(lag_map[i, j, k])

    y_raw = bold_data[i, j, k, :].astype(np.float64)

    y_mean = np.nanmean(y_raw)
    if not np.isfinite(y_mean) or y_mean == 0:
        continue

    y_percent = 100.0 * (y_raw / y_mean - 1.0)

    x_shifted = shift_regressor_by_lag(
        gm_global_percent,
        t,
        lag_sec,
        sign=LAG_SIGN
    )

    beta, intercept, r, r2, t_beta, n_valid, resid_std = fit_simple_glm(
        y_percent,
        x_shifted,
        min_valid=MIN_VALID
    )

    beta_map[i, j, k] = beta
    intercept_map[i, j, k] = intercept
    r_map[i, j, k] = r
    r2_map[i, j, k] = r2
    t_map[i, j, k] = t_beta
    nvalid_map[i, j, k] = n_valid
    residstd_map[i, j, k] = resid_std

print("GLM finished.")

BOLD shape: (79, 95, 79, 600)
Stable lag voxels: 98394
LAG_SIGN: -1
Saved regressor: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-GMGlobalMean_percent_regressor.txt
Regressor mean: 7.832845e-06
Regressor std : 0.25573796
Running GLM voxels: 98394
MIN_VALID: 420
LAG_SIGN: -1
0/98394
5000/98394
10000/98394
15000/98394
20000/98394
25000/98394
30000/98394
35000/98394
40000/98394
45000/98394
50000/98394
55000/98394
60000/98394
65000/98394
70000/98394
75000/98394
80000/98394
85000/98394
90000/98394
95000/98394
GLM finished.


In [16]:
# SAVE
beta_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_beta_map.nii.gz"
intercept_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_intercept_map.nii.gz"
r_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_R_map.nii.gz"
r2_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_R2_map.nii.gz"
t_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_t_map.nii.gz"
nvalid_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_nvalid_map.nii.gz"
residstd_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_residstd_map.nii.gz"
usedmask_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_usedMask.nii.gz"

save_float_map(beta_map, bold_img, beta_path)
save_float_map(intercept_map, bold_img, intercept_path)
save_float_map(r_map, bold_img, r_path)
save_float_map(r2_map, bold_img, r2_path)
save_float_map(t_map, bold_img, t_path)
save_float_map(nvalid_map, bold_img, nvalid_path)
save_float_map(residstd_map, bold_img, residstd_path)
save_mask(stable_lag_mask, bold_img, usedmask_path)

print("Saved:")
print(beta_path)
print(r_path)
print(r2_path)
print(t_path)
print(nvalid_path)
print(residstd_path)
print(usedmask_path)


# Save run metadata
glm_metadata = {
    "subject": SUBJECT,
    "description": "Lag-corrected rsCVR-like GLM using internal GM global BOLD regressor",
    "not_calibrated_CVR": True,
    "unit_interpretation": "% voxel BOLD change per % GM global signal change",
    "TR": TR,
    "LAG_SIGN": LAG_SIGN,
    "search_range_narrow": SEARCH_RANGE_NARROW,
    "search_range_wide": SEARCH_RANGE_WIDE,
    "stable_lag_mask_criteria": {
        "lag_diff_abs_leq_sec": LAG_DIFF_THR,
        "lag45_min_sec": STABLE_LAG_MIN,
        "lag45_max_sec": STABLE_LAG_MAX,
        "maxcorr45_geq": MAXCORR_THR,
        "mtt45_min_sec": MTT_MIN,
        "mtt45_max_sec": MTT_MAX,
    },
    "R2_threshold_for_QC": R2_THR,
    "bold4d_path": str(bold4d_path),
    "gm_mask_path": str(gm_mask_path),
    "stable_lag_mask_path": str(stable_lag_mask_path),
    "lag_map_used": str(lag_path),
    "gm_global_regressor": str(reg_path),
}

metadata_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(glm_metadata, f, indent=2)

print("Metadata:", metadata_path)

Saved:
/media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlagGLM_beta_map.nii.gz
/media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlagGLM_R_map.nii.gz
/media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlagGLM_R2_map.nii.gz
/media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlagGLM_t_map.nii.gz
/media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlagGLM_nvalid_map.nii.gz
/media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlagGLM_residstd_map.nii.gz
/media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlag

In [17]:
# GLM map statistics and R2 QC mask
def stat_map_from_array(data, mask, name):
    vals = data[mask]
    vals = vals[np.isfinite(vals)]

    print("\n" + "=" * 80)
    print(name)
    print("n:", vals.size)

    if vals.size == 0:
        return

    print("min/max:", vals.min(), vals.max())
    for p in [0.1, 1, 2, 5, 25, 50, 75, 95, 98, 99, 99.9]:
        print(f"p{p}: {np.percentile(vals, p)}")


stat_map_from_array(beta_map, stable_lag_mask, "rsCVR lag-GLM beta")
stat_map_from_array(r_map, stable_lag_mask, "rsCVR lag-GLM R")
stat_map_from_array(r2_map, stable_lag_mask, "rsCVR lag-GLM R2")
stat_map_from_array(t_map, stable_lag_mask, "rsCVR lag-GLM t")
stat_map_from_array(nvalid_map, stable_lag_mask, "n valid timepoints")


# R2 QC mask
matter = np.squeeze(nib.load(str(matter_mask_path)).get_fdata()) > 0

glm_qc_mask = stable_lag_mask & np.isfinite(beta_map) & (r2_map >= R2_THR)

glm_qc_mask_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_R2ge{R2_THR:.2f}_mask.nii.gz"
save_mask(glm_qc_mask, bold_img, glm_qc_mask_path)

beta_qc = beta_map.copy()
beta_qc[~glm_qc_mask] = np.nan

beta_qc_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_beta_R2ge{R2_THR:.2f}_map.nii.gz"
save_float_map(beta_qc, bold_img, beta_qc_path)

print("\nGLM QC voxels:", int(glm_qc_mask.sum()))
print("GLM QC / stable lag:", glm_qc_mask.sum() / stable_lag_mask.sum())
print("GLM QC / matter:", glm_qc_mask.sum() / matter.sum())

stat_map_from_array(beta_map, glm_qc_mask, f"beta | R2 >= {R2_THR}")
stat_map_from_array(r2_map, glm_qc_mask, f"R2 | R2 >= {R2_THR}")

print("Saved QC mask:", glm_qc_mask_path)
print("Saved beta QC map:", beta_qc_path)


rsCVR lag-GLM beta
n: 97019
min/max: -195.8473 392.7075
p0.1: -9.48210334777832
p1: -3.9558472633361816
p2: -2.7378361225128174
p5: -1.4773670434951782
p25: 0.03954290598630905
p50: 0.8705444931983948
p75: 1.744844675064087
p95: 3.6138014793395996
p98: 5.032959461212158
p99: 6.777434825897217
p99.9: 17.67099380493164

rsCVR lag-GLM R
n: 97019
min/max: -0.5863398 0.7402708
p0.1: -0.4318130314350128
p1: -0.33295997977256775
p2: -0.294676274061203
p5: -0.22224931418895721
p25: 0.00821942649781704
p50: 0.19809488952159882
p75: 0.3633521795272827
p95: 0.5385891795158386
p98: 0.589770495891571
p99: 0.6196908950805664
p99.9: 0.6924868226051331

rsCVR lag-GLM R2
n: 97019
min/max: 2.0775987e-12 0.5480008
p0.1: 1.975827217393089e-07
p1: 1.8729861039901152e-05
p2: 7.439294859068468e-05
p5: 0.0004638034733943641
p25: 0.011966065503656864
p50: 0.053045306354761124
p75: 0.13451966643333435
p95: 0.2901003658771515
p98: 0.3478291928768158
p99: 0.38401687145233154
p99.9: 0.4795379638671875

rsCVR lag-

In [18]:
# Viewer: GLM rsCVR-like maps

glm_map_dict = {
    "beta": beta_path,
    "beta_R2QC": beta_qc_path,
    "R": r_path,
    "R2": r2_path,
    "t": t_path,
    "nvalid": nvalid_path,
    "residstd": residstd_path,
    "usedMask": usedmask_path,
    "R2QCmask": glm_qc_mask_path,
}

bold_img = nib.load(str(bold4d_path))
bold_data = bold_img.get_fdata(dtype=np.float32)
nx, ny, nz, nt = bold_data.shape

used_mask = np.squeeze(nib.load(str(usedmask_path)).get_fdata()) > 0


def load_glm_map(name):
    return np.squeeze(nib.load(str(glm_map_dict[name])).get_fdata(dtype=np.float32))


def show_glm_map(
    map_name="beta",
    volume=1,
    z=nz//2,
    alpha=0.65,
    bg_vmin_pct=1,
    bg_vmax_pct=99,
    map_low_pct=1,
    map_high_pct=99,
    use_mask=True,
    symmetric=False
):
    bg = bold_data[..., volume - 1]
    data = load_glm_map(map_name)

    if use_mask and map_name not in ["usedMask", "R2QCmask", "beta_R2QC"]:
        data = data.copy()
        data[~used_mask] = np.nan

    bg2d = bg[:, :, z].T
    map2d = data[:, :, z].T

    bg_vmin, bg_vmax = np.percentile(bg[np.isfinite(bg)], [bg_vmin_pct, bg_vmax_pct])

    vals = data[np.isfinite(data)]
    vals = vals[vals != 0]

    if vals.size == 0:
        map_vmin, map_vmax = -1, 1
    else:
        map_vmin, map_vmax = np.percentile(vals, [map_low_pct, map_high_pct])

    if map_name in ["R2", "nvalid", "residstd", "usedMask", "R2QCmask"]:
        symmetric = False

    if symmetric:
        vmax_abs = max(abs(map_vmin), abs(map_vmax))
        map_vmin, map_vmax = -vmax_abs, vmax_abs

    map2d_plot = np.ma.masked_where((~np.isfinite(map2d)) | (map2d == 0), map2d)

    plt.figure(figsize=(7, 7))
    plt.imshow(bg2d, cmap="gray", origin="lower", vmin=bg_vmin, vmax=bg_vmax)

    if map_name in ["beta", "beta_R2QC", "R", "t"]:
        cmap = "coolwarm"
    elif map_name in ["usedMask", "R2QCmask"]:
        cmap = "autumn"
    else:
        cmap = "magma"

    im = plt.imshow(
        map2d_plot,
        cmap=cmap,
        origin="lower",
        alpha=alpha,
        vmin=map_vmin,
        vmax=map_vmax
    )

    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title(
        f"{SUBJECT} | lag-corrected GLM rsCVR-like map\n"
        f"{map_name} | BG volume {volume}/{nt} | Axial z={z}"
    )
    plt.axis("off")
    plt.show()


interact(
    show_glm_map,
    map_name=Dropdown(
        options=list(glm_map_dict.keys()),
        value="beta",
        description="Map"
    ),
    volume=IntSlider(value=1, min=1, max=nt, step=1, description="BG vol"),
    z=IntSlider(value=nz//2, min=0, max=nz-1, step=1, description="Z slice"),
    alpha=FloatSlider(value=0.65, min=0.0, max=1.0, step=0.05, description="Alpha"),
    bg_vmin_pct=FloatSlider(value=1, min=0, max=10, step=0.5, description="BG min%"),
    bg_vmax_pct=FloatSlider(value=99, min=90, max=100, step=0.5, description="BG max%"),
    map_low_pct=FloatSlider(value=1, min=0, max=20, step=0.5, description="Map low%"),
    map_high_pct=FloatSlider(value=99, min=80, max=100, step=0.5, description="Map high%"),
    use_mask=Checkbox(value=True, description="stable mask only"),
    symmetric=Checkbox(value=True, description="symmetric")
)

interactive(children=(Dropdown(description='Map', options=('beta', 'beta_R2QC', 'R', 'R2', 't', 'nvalid', 'res…

<function __main__.show_glm_map(map_name='beta', volume=1, z=39, alpha=0.65, bg_vmin_pct=1, bg_vmax_pct=99, map_low_pct=1, map_high_pct=99, use_mask=True, symmetric=False)>

In [19]:
# Save summary CSV
def percentile_summary(data, mask, prefix):
    vals = data[mask]
    vals = vals[np.isfinite(vals)]

    if vals.size == 0:
        return {}

    return {
        f"{prefix}_n": int(vals.size),
        f"{prefix}_p1": float(np.percentile(vals, 1)),
        f"{prefix}_p5": float(np.percentile(vals, 5)),
        f"{prefix}_p50": float(np.percentile(vals, 50)),
        f"{prefix}_p95": float(np.percentile(vals, 95)),
        f"{prefix}_p99": float(np.percentile(vals, 99)),
    }


matter = np.squeeze(nib.load(str(matter_mask_path)).get_fdata()) > 0

summary = {
    "subject": SUBJECT,
    "TR_sec": TR,
    "n_volumes": int(nt),
    "brain_voxels": int(brain_mask.sum()),
    "matter_voxels": int(matter.sum()),
    "gm_voxels": int(gm_mask.sum()),
    "stable_lag_voxels": int(stable_lag_mask.sum()),
    "stable_lag_over_matter": float(stable_lag_mask.sum() / matter.sum()),
    "glm_qc_voxels": int(glm_qc_mask.sum()),
    "glm_qc_over_stable_lag": float(glm_qc_mask.sum() / stable_lag_mask.sum()),
    "glm_qc_over_matter": float(glm_qc_mask.sum() / matter.sum()),
    "LAG_SIGN": LAG_SIGN,
    "R2_THR": R2_THR,
    "bold4d_path": str(bold4d_path),
    "stable_lag_mask_path": str(stable_lag_mask_path),
    "beta_path": str(beta_path),
    "beta_qc_path": str(beta_qc_path),
    "r2_path": str(r2_path),
    "t_path": str(t_path),
}

summary.update(percentile_summary(beta_map, stable_lag_mask, "beta_stable"))
summary.update(percentile_summary(r2_map, stable_lag_mask, "R2_stable"))
summary.update(percentile_summary(t_map, stable_lag_mask, "t_stable"))

summary.update(percentile_summary(beta_map, glm_qc_mask, "beta_R2QC"))
summary.update(percentile_summary(r2_map, glm_qc_mask, "R2_R2QC"))
summary.update(percentile_summary(t_map, glm_qc_mask, "t_R2QC"))

summary_csv_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_summary.csv"
summary_json_path = glm_out_dir / f"{SUBJECT}_desc-rsCVRlagGLM_summary.json"

pd.DataFrame([summary]).to_csv(summary_csv_path, index=False)

with open(summary_json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved summary CSV :", summary_csv_path)
print("Saved summary JSON:", summary_json_path)

Saved summary CSV : /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlagGLM_summary.csv
Saved summary JSON: /media/bami/JHA/rsCVR/data/Tests/Test_1a/RapidTide/rsCVR_lagGLM_internalGMprobe_search-15to45_stableLag/Test_1a_desc-rsCVRlagGLM_summary.json


In [20]:
# Final interpretation note

print("""
Final output interpretation

Primary map:
  *_desc-rsCVRlagGLM_beta_map.nii.gz

QC-restricted primary map:
  *_desc-rsCVRlagGLM_beta_R2ge0.05_map.nii.gz

Meaning:
  lag-corrected rsCVR-like beta map
  = % voxel BOLD change / % GM global signal change

Important:
  This is NOT calibrated CO2-CVR.
  This is NOT %BOLD/mmHg CO2.
  This is an internal GM global BOLD signal-derived, lag-corrected, resting-state CVR-like response map.

Recommended maps to inspect:
  beta
  beta_R2QC
  R2
  t
  usedMask
  R2QCmask
  maxtimerefined_map from rapidtide
  maxcorr_map from rapidtide
""")


Final output interpretation

Primary map:
  *_desc-rsCVRlagGLM_beta_map.nii.gz

QC-restricted primary map:
  *_desc-rsCVRlagGLM_beta_R2ge0.05_map.nii.gz

Meaning:
  lag-corrected rsCVR-like beta map
  = % voxel BOLD change / % GM global signal change

Important:
  This is NOT calibrated CO2-CVR.
  This is NOT %BOLD/mmHg CO2.
  This is an internal GM global BOLD signal-derived, lag-corrected, resting-state CVR-like response map.

Recommended maps to inspect:
  beta
  beta_R2QC
  R2
  t
  usedMask
  R2QCmask
  maxtimerefined_map from rapidtide
  maxcorr_map from rapidtide

